# 第 00 关：搭建数据实验台

**本关要回答的问题：** 资产定价研究用的数据长什么样？怎样把下载的文件整理成可以直接做回归的样本？

做完本关，你会得到两张按月份对齐的表：一张是**因子收益**（回归的解释变量），一张是**测试资产的超额收益**（回归的被解释变量）。第 02、03、05 关都使用这份样本。它对应 Fama–French（1993，下称 FF1993）的研究期间：1963 年 7 月至 1991 年 12 月，共 342 个月。

### 先认识几个概念
- **收益率**：持有一个月的回报，等于（月末价格 + 当月分红）÷ 月初价格 − 1。原始文件用百分数记录，例如 2.89 表示 2.89%。
- **无风险利率 RF**：同一个月持有 1 个月期美国国库券的收益，代表"不承担风险也能拿到的回报"。
- **超额收益**：资产收益 − RF，即承担风险之后多赚的部分。资产定价模型解释的是超额收益，因为它才是风险的报酬。
- **市场超额收益 Mkt-RF**：全部美国上市股票按市值加权的组合收益减去 RF，代表"整个股市"的回报。
- **因子**：每月一个数的收益序列，代表许多股票共同面对的某种风险来源。三因子文件里除了 Mkt-RF，还有**规模因子 SMB**（小公司股票组合收益减大公司股票组合收益）和**价值因子 HML**（高账面市值比股票减低账面市值比股票）。它们的构造在第 04 关详细讲。
- **测试资产**：用来检验模型的资产。这里是 25 个股票组合：按公司市值（规模）分五档，再按**账面市值比 B/M**（账面权益 ÷ 市值，衡量股价相对账面价值是"便宜"还是"贵"）分五档，交叉得到 5×5 = 25 个组合。用组合而不用单只股票，是因为组合把个股的特有噪声平均掉了，收益规律更清楚。

**为什么要认真整理数据？** 回归要求左右两边的每一行是同一个月、单位一致。月份错开一行，或把百分数当成小数，代码照样能算出数字，但这些数字已经在回答另一个问题了。

第一个代码格找到项目目录，并导入课程的公共函数（`src/course.py` 负责读数据和作图，`src/analysis.py` 负责回归与检验）。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/course.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from course import (load_raw, load_returns, raw_path, catalog, manifest, style,
                    build_ff3, build_ff5, heatmap25, save_table, paper_note)
from analysis import fit_models, grs_test, model_summary
style()
print('项目目录：', ROOT)

项目目录： E:\Eth data\asset_pricing_quest


In [2]:
display(Markdown(paper_note('ff1993')))

**论文对照：** [Fama & French (1993), Common risk factors in the returns on stocks and bonds](../papers/ff1993_primary_reference.pdf)。准确表号、PDF 页码和复现范围见 [论文表格导航](../papers/TABLE_GUIDE.md)。当前数据版本会与原文历史版本存在差异。

## 1. 查看原文件与来源
先读文件开头的说明文字。它写明了数据版本（用哪个月的 CRSP 数据库生成；CRSP 是芝加哥大学证券价格研究中心维护的美国股票数据库）、RF 的来源，以及组合的构造方法。`manifest()` 另外记录了每个文件的下载时间和 SHA-256 校验值，保证每次运行用的是同一份数据。

In [3]:
for key in ['ff3','bm25']:
    record = manifest()['datasets'][key]
    print(key, record['source_header'], record['downloaded_at_utc'])
    print('\n'.join(raw_path(key).read_text(encoding='utf-8-sig').splitlines()[:9]))
    print()

ff3 This file was created using the 202609 CRSP database. 2026-11-08T02:40:46+00:00
This file was created using the 202609 CRSP database.
The 1-month TBill rate data until 202405 are from Ibbotson Associates. Starting from 202406, the 1-month TBill rate is from ICE BofA US 1-Month Treasury Bill Index. 
The annual TBill return is compounded from the monthly T-bill rates from January to December. 

,Mkt-RF,SMB,HML,RF
192607,   2.89,  -2.42,  -2.75,   0.22
192608,   2.64,  -1.44,   4.13,   0.25
192609,   0.38,  -1.36,   0.05,   0.23
192610,  -3.27,  -0.18,   0.78,   0.32

bm25 This file was created using the 202609 CRSP database. 2026-11-08T02:40:49+00:00
This file was created using the 202609 CRSP database.
It contains value- and equal-weighted returns for portfolios formed on ME and BEME.

The portfolios are constructed at the end of June.  BEME is book value at the last fiscal year end of the prior calendar year
divided by ME at the end of December of the prior year.
Annual returns are

**读数：** 25 组合文件的说明交代了后面会用到的三条规则：
1. 组合在每年 6 月底形成；
2. BEME（即 B/M）= 上一财年末的账面价值 ÷ 上一年 12 月底的市值；
3. 缺失值记为 -99.99 或 -999。

## 2. 读取两张月度表
原文件里有好几张表（月度收益、年度收益、等权收益、市值加权收益……）。`load_raw` 找到其中"月度、市值加权收益"这一张，原样读成字符串，暂时不做任何换算。具体定位条件见 `src/course.py`。

下面三张输出依次是：两张表的行数、列数和起止月份；因子表的前 3 行；25 组合表前 3 行的前 5 列。组合的列名中，ME 指市值、BM 指账面市值比：`SMALL LoBM` 是"规模最小、B/M 最低"，`ME1 BM2` 是"规模第 1 档、B/M 第 2 档"。

In [4]:
ff_raw = load_raw('ff3')
p_raw = load_raw('bm25')
display(pd.DataFrame({'rows':[len(ff_raw),len(p_raw)],'columns':[ff_raw.shape[1],p_raw.shape[1]],'first':[ff_raw.index[0],p_raw.index[0]],'last':[ff_raw.index[-1],p_raw.index[-1]]},index=['factors','portfolios']))
display(ff_raw.head(3))
display(p_raw.iloc[:3,:5])

,rows,columns,first,last
factors,1203,4,192607,202609
portfolios,1203,25,192607,202609


,Mkt-RF,SMB,HML,RF
date_raw,,,,
192607,2.89,-2.42,-2.75,0.22
192608,2.64,-1.44,4.13,0.25
192609,0.38,-1.36,0.05,0.23


,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM
date_raw,,,,,
192607,3.7806,-0.4119,-1.9223,-0.4114,1.9583
192608,-2.2074,-8.7275,2.4557,1.3500,8.5104
192609,-6.2460,-0.3260,-6.1982,-1.6318,0.8586


## 3. 整理成研究样本
这一步做三件事：
1. **转成数字，标记缺失值**：把 -99.99 和 -999 换成缺失值 `NaN`。
2. **统一单位**：除以 100，把百分数变成小数（2.89 → 0.0289）。计算统一用小数，展示时再乘 100。
3. **统一日期**：把 `196307` 这样的整数变成月度 `PeriodIndex`，含义是"1963 年 7 月这个月"，而不是某个具体日期。

然后截取 FF1993 的样本期 1963-07 至 1991-12。最后的检查表逐项确认：月份数为 342、列数正确、两张表的月份完全一致、中间没有缺月、没有重复月份、没有缺失值。任何一项为 `False`，`assert` 都会让程序停下来。

In [5]:
START, END = '1963-07', '1991-12'
def prepare(raw):
    frame = raw.apply(pd.to_numeric, errors='raise')
    frame = frame.replace([-99.99, -999], np.nan).astype(float) / 100
    frame.index = pd.PeriodIndex(pd.to_datetime(raw.index, format='%Y%m'), freq='M', name='month')
    return frame.loc[START:END].copy()
ff = prepare(ff_raw)
portfolios = prepare(p_raw)
expected = pd.period_range(START, END, freq='M', name='month')
checks = pd.Series({
    '342 months': len(ff)==len(portfolios)==342,
    '4 factor-file columns': ff.shape[1]==4,
    '25 test portfolios': portfolios.shape[1]==25,
    'same monthly index': ff.index.equals(portfolios.index),
    'complete calendar': ff.index.equals(expected),
    'unique dates': ff.index.is_unique and portfolios.index.is_unique,
    'no missing values': not(ff.isna().any().any() or portfolios.isna().any().any()),
})
display(checks.to_frame('passed'))
assert checks.all()

,passed
342 months,True
4 factor-file columns,True
25 test portfolios,True
same monthly index,True
complete calendar,True
unique dates,True
no missing values,True


## 4. 构造回归要用的两个对象
- `test_excess`：25 个组合的收益逐月减去当月 RF，得到超额收益，放在回归的**左边**。
- `factors`：Mkt-RF、SMB、HML 三列，放在回归的**右边**。Mkt-RF 已经减过 RF；SMB、HML 是两组股票收益之差，RF 在相减时已经抵消，所以三列都不再减 RF。

两张表保存到 `results/` 目录，方便以后查看。

In [6]:
test_excess = portfolios.sub(ff['RF'], axis=0)
factors = ff[['Mkt-RF','SMB','HML']]
display(test_excess.iloc[:3,:5])
save_table(test_excess, '00_test_excess_1963_1991')
save_table(factors, '00_factors_1963_1991')

,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM
month,,,,,
1963-07,0.0086,-0.0063,0.0045,-0.0031,-0.0151
1963-08,0.0399,0.0112,0.0124,0.0226,0.0441
1963-09,-0.0200,0.0035,-0.0127,-0.0179,-0.0063


WindowsPath('E:/Eth data/asset_pricing_quest/results/00_factors_1963_1991.csv')

## 怎样读这些输出
- 25 列各是一个股票组合的月度超额收益历史，不是单个公司。例如 1963-07 那一行 `SMALL LoBM` 的 0.0086 表示：这个组合当月比国库券多赚了 0.86%。
- 单月收益波动很大，既有全市场共同的涨跌，也有组合自己的噪声。后面各关关心的是**长期平均收益**在组合之间的差异，以及这种差异能否用风险来解释。
- 本关只做了整理，还没有检验任何模型。

**通关提交：** 上面的检查表；用自己的话说明因子文件的 4 列（Mkt-RF、SMB、HML、RF）和任意一个测试组合列各是什么经济量；以及任何你还说不清楚的处理步骤。

## Boss 作答区

四道数据题。先写出自己的推理，再动手改代码验证。程序没有报错，并不说明数据处理正确。

1. 如果忘了把百分数除以 100，但左右两边都忘了，回归的 β 和 α 各会怎样变化？如果只有一边忘了呢？
2. 如果因子表比组合表错开了一个月，β 会变大还是变小？为什么？
3. 为什么测试组合要减去 RF，而 SMB、HML 不减？
4. 如果某个 -99.99 没有被换成缺失值，会对该组合的平均收益造成多大影响？

**我的回答：**

（由你填写；本课程未预填答案。）